# GrowAI — Dataset Pre-Processing Notebook
**Thullen Lab | Chukwuasia Madike & Sreytouch Jessica Lang | July 2026**

This notebook covers pre-processing for all 5 confirmed GrowAI datasets:
1. PlantVillage (image CNN data)
2. USDA NASS Quick Stats (tabular/API)
3. Open-Meteo (weather API)
4. Kaggle Agriculture & Farming (tabular CSV)
5. Indoor House Plants (text/CSV knowledge base)


---
## 1. PlantVillage — Image Pre-Processing

In [ ]:
# ── PlantVillage Image Pre-Processing ──
# Dataset: 54,306 images, 38 disease/species classes
# Download from: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
# Expected folder structure: plantvillage/<ClassName>/<image>.jpg

from PIL import Image
import imagehash
from torchvision import transforms
from collections import defaultdict

PLANTVILLAGE_DIR = DATA_DIR / 'plantvillage'
OUTPUT_DIR = PROCESSED_DIR / 'plantvillage'

# Home-garden-relevant classes (filter out industrial-only crops)
HOME_GARDEN_CLASSES = [
    'Tomato', 'Pepper', 'Potato', 'Strawberry', 'Peach',
    'Grape', 'Apple', 'Blueberry', 'Cherry', 'Raspberry',
    'Squash', 'Corn', 'Soybean'  # remove if not relevant to your region
]

# Step 1: Scan classes available
if PLANTVILLAGE_DIR.exists():
    all_classes = [d.name for d in PLANTVILLAGE_DIR.iterdir() if d.is_dir()]
    filtered_classes = [c for c in all_classes if any(hg in c for hg in HOME_GARDEN_CLASSES)]
    print(f'Total classes: {len(all_classes)}')
    print(f'Home-garden classes after filter: {len(filtered_classes)}')
    print(filtered_classes[:10], '...')
else:
    all_classes = []
    filtered_classes = []
    print(f'PlantVillage folder not found: {PLANTVILLAGE_DIR}')
    print('Add the dataset there before running this section.')

KeyboardInterrupt: 

In [ ]:
# ── Install dependencies (run once) ──
# !pip install pandas numpy scikit-learn pillow imagehash requests tqdm torchvision

import os, json, warnings
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from tqdm import tqdm

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / '.git').exists() and (path / 'src').exists()),
    Path.cwd()
)
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

warnings.filterwarnings('ignore')
print(f'Project root: {PROJECT_ROOT}')
print('Libraries loaded ✓')

In [ ]:
# Step 2: Resize + Normalize transforms (ImageNet mean/std for MobileNetV3)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Augmentation pipeline (for training set only)
augment = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print('Transforms defined ✓')

In [ ]:
# Step 3: Deduplicate using perceptual hash
def deduplicate_images(image_paths, threshold=8):
    """Remove near-duplicate images using perceptual hash."""
    seen_hashes = {}
    unique_paths = []
    duplicates = []
    for path in tqdm(image_paths, desc='Deduplicating'):
        try:
            img = Image.open(path).convert('RGB')
            h = imagehash.phash(img)
            is_dup = any(abs(h - seen) <= threshold for seen in seen_hashes)
            if not is_dup:
                seen_hashes[h] = path
                unique_paths.append(path)
            else:
                duplicates.append(path)
        except Exception as e:
            print(f'Error processing {path}: {e}')
    print(f'Removed {len(duplicates)} duplicates, kept {len(unique_paths)} unique images')
    return unique_paths

# Example usage (uncomment when running with actual dataset)
# all_images = list(PLANTVILLAGE_DIR.rglob('*.jpg'))
# unique_images = deduplicate_images(all_images)
print('Deduplication function ready ✓')

In [ ]:
# Step 4: Train/Val/Test split (stratified by class)
from sklearn.model_selection import train_test_split

def stratified_split(image_paths, labels, train=0.70, val=0.15, test=0.15):
    """70/15/15 stratified split by class label."""
    X_train, X_temp, y_train, y_temp = train_test_split(
        image_paths, labels, test_size=(1 - train), stratify=labels, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(test / (val + test)), stratify=y_temp, random_state=42)
    print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
    return X_train, X_val, X_test, y_train, y_val, y_test

print('Stratified split function ready ✓')

In [ ]:
from collections import Counter

all_images = [
    p for p in PLANTVILLAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
print("All matched images:", len(all_images))

raw_counts = Counter(p.parent.name for p in all_images)
rare_classes = {name: count for name, count in raw_counts.items() if count < 5}
print("Rare classes removed:", rare_classes)

filtered_images = [p for p in all_images if raw_counts[p.parent.name] >= 5]

unique_images = deduplicate_images(filtered_images)
labels = [img.parent.name for img in unique_images]

post_counts = Counter(labels)
unique_images = [img for img in unique_images if post_counts[img.parent.name] >= 5]
labels = [img.parent.name for img in unique_images]

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(unique_images, labels)

print("Unique images:", len(unique_images))
print("Classes kept:", len(set(labels)))
print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

---
## 2. USDA NASS Quick Stats — Tabular Pre-Processing

In [ ]:
# ── USDA NASS Quick Stats ──
# API Docs: https://quickstats.nass.usda.gov/api
# Register for a free API key at: https://quickstats.nass.usda.gov/api

USDA_API_KEY = 'YOUR_API_KEY_HERE'   # Replace with your registered key
USDA_BASE_URL = 'https://quickstats.nass.usda.gov/api/api_GET/'

# HOME_GARDEN_CROPS = [
#     'TOMATOES', 'PEPPERS', 'LETTUCE', 'BEANS', 'CUCUMBERS',
#     'SQUASH', 'HERBS', 'STRAWBERRIES'
# ]
HOME_GARDEN_CROPS = [
    'TOMATOES', 'PEPPERS', 'LETTUCE', 'BEANS',
    'CUCUMBERS', 'SQUASH', 'STRAWBERRIES'
]

def fetch_usda_nass(commodity, api_key=USDA_API_KEY):
    """Fetch USDA NASS Quick Stats for a commodity."""
    params = {
        'key': api_key,
        'commodity_desc': commodity,
        'statisticcat_desc': 'YIELD',
        'year__GE': '2015',
        'format': 'JSON'
    }
    response = requests.get(USDA_BASE_URL, params=params)
    if response.status_code == 200:
        data = response.json()
        return pd.DataFrame(data.get('data', []))
    else:
        print(f'Error fetching {commodity}: {response.status_code}')
        return pd.DataFrame()

print('USDA NASS fetch function ready ✓')
print('Remember to set your USDA_API_KEY before running!')

In [ ]:
def preprocess_usda(df):
    """Full pre-processing pipeline for USDA NASS data."""
    print(f'Raw shape: {df.shape}')

    # Step 1: Keep relevant columns
    keep_cols = ['year', 'state_name', 'commodity_desc', 'statisticcat_desc',
             'unit_desc', 'Value', 'CV (%)']
    df = df[[c for c in keep_cols if c in df.columns]].copy()

    # Step 2: Standardize column names
    df.columns = [c.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('%', 'pct') for c in df.columns]

    # Step 3: Drop rows with missing value or state
    df = df.dropna(subset=['value', 'state_name'])
    df = df[df['value'] != ' (D)']   # (D) = withheld to avoid disclosure

    # Step 4: Convert types
    df['value'] = pd.to_numeric(df['value'].str.replace(',', ''), errors='coerce')
    df['year']  = pd.to_numeric(df['year'], errors='coerce')
    df['state_name']    = df['state_name'].astype('category')
    df['commodity_desc'] = df['commodity_desc'].astype('category')

    # Step 5: Drop columns with >30% missing
    thresh = 0.7
    df = df.dropna(thresh=int(thresh * len(df)), axis=1)

    # Step 6: Feature engineering
    df = df.sort_values(['commodity_desc', 'state_name', 'year'])
    df['yield_3yr_avg'] = (df.groupby(['commodity_desc', 'state_name'])['value']
                           .transform(lambda x: x.rolling(3, min_periods=1).mean()))
    df['yield_trend'] = df.groupby(['commodity_desc', 'state_name'])['value'].transform(
        lambda x: x.diff().rolling(3, min_periods=1).mean())

    # Step 7: Normalize
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    df['value_norm'] = scaler.fit_transform(df[['value']])

    print(f'Processed shape: {df.shape}')
    print(df.dtypes)
    return df

# Example (uncomment when API key is set):
# raw_dfs = [fetch_usda_nass(crop) for crop in HOME_GARDEN_CROPS]
# usda_raw = pd.concat(raw_dfs, ignore_index=True)
# usda_clean = preprocess_usda(usda_raw)
# usda_clean.to_csv('data/processed/usda_nass_clean.csv', index=False)
# usda_clean.to_parquet('data/processed/usda_nass_clean.parquet', index=False)
print('USDA NASS pre-processing function ready ✓')

In [ ]:
raw_dfs = [fetch_usda_nass(crop, api_key=USDA_API_KEY) for crop in HOME_GARDEN_CROPS]
non_empty_dfs = [df for df in raw_dfs if not df.empty]

if not non_empty_dfs:
    print("USDA fetch failed. Check your API key and rerun the USDA function cell.")
else:
    usda_raw = pd.concat(non_empty_dfs, ignore_index=True)
    usda_clean = preprocess_usda(usda_raw)
    data_quality_report(usda_clean, "USDA NASS Quick Stats")
    usda_clean.head()

---
## 3. Open-Meteo — Weather API Pre-Processing

In [ ]:
# ── Open-Meteo Weather API ──
# No API key required for non-commercial use
# Docs: https://open-meteo.com/en/docs

def fetch_weather(lat, lon, days_forecast=7, days_history=90):
    """Fetch forecast + historical weather for a location."""
    from datetime import datetime, timedelta

    end_date   = datetime.today().strftime('%Y-%m-%d')
    start_date = (datetime.today() - timedelta(days=days_history)).strftime('%Y-%m-%d')

    # Historical data
    hist_url = 'https://archive-api.open-meteo.com/v1/archive'
    hist_params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start_date, 'end_date': end_date,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration',
        'timezone': 'auto'
    }

    # Forecast data
    fcast_url = 'https://api.open-meteo.com/v1/forecast'
    fcast_params = {
        'latitude': lat, 'longitude': lon,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration,uv_index_max',
        'forecast_days': days_forecast,
        'timezone': 'auto'
    }

    hist_resp  = requests.get(hist_url, params=hist_params).json()
    fcast_resp = requests.get(fcast_url, params=fcast_params).json()

    hist_df  = pd.DataFrame(hist_resp.get('daily', {}))
    fcast_df = pd.DataFrame(fcast_resp.get('daily', {}))

    combined = pd.concat([hist_df, fcast_df], ignore_index=True)
    combined['lat'] = lat
    combined['lon'] = lon
    return combined


def preprocess_weather(df, watering_threshold_mm=5.0):
    """Clean and engineer features from raw Open-Meteo data."""
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)

    # Handle nulls with forward-fill (up to 2 days)
    df = df.fillna(method='ffill', limit=2)
    df['data_gap_flag'] = df.isnull().any(axis=1)

    # Derived watering indicator
    df['et0']   = df.get('et0_fao_evapotranspiration', 0)
    df['precip'] = df.get('precipitation_sum', 0)
    df['water_deficit_mm'] = (df['et0'] - df['precip']).clip(lower=0)
    df['water_needed']     = df['water_deficit_mm'] > watering_threshold_mm

    # Rolling 7-day averages
    for col in ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']:
        if col in df.columns:
            df[f'{col}_7d_avg'] = df[col].rolling(7, min_periods=1).mean()

    print(f'Weather data shape: {df.shape}')
    print(f'Date range: {df["time"].min()} → {df["time"].max()}')
    print(f'Days needing water: {df["water_needed"].sum()} / {len(df)}')
    return df


# Example: Long Beach, CA (Jessica's location)
# weather_raw = fetch_weather(lat=33.77, lon=-118.19)
# weather_clean = preprocess_weather(weather_raw)
# weather_clean.to_csv('data/processed/weather_longbeach.csv', index=False)
print('Open-Meteo pre-processing functions ready ✓')

---
## 4. Kaggle Agriculture & Farming — Tabular Pre-Processing

In [ ]:
# ── Kaggle Agriculture & Farming ──
# Download from Kaggle (requires kaggle CLI or manual download)
# !kaggle datasets download -d atharvaingle/crop-recommendation-dataset

def preprocess_kaggle_agri(filepath):
    """Pre-process Kaggle crop recommendation / agriculture dataset."""
    df = pd.read_csv(filepath)
    print(f'Raw shape: {df.shape}')
    print(df.head(2))

    # Step 1: Standardize column names
    df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]

    # Step 2: Drop columns with >40% missing
    missing_pct = df.isnull().mean()
    df = df.loc[:, missing_pct < 0.40]
    print(f'After dropping high-missing columns: {df.shape}')

    # Step 3: Impute remaining numerics with median
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # Step 4: Standardize crop/label column
    label_col = [c for c in df.columns if 'label' in c or 'crop' in c]
    if label_col:
        df[label_col[0]] = df[label_col[0]].str.lower().str.strip()

    # Step 5: Filter to home-garden-relevant crops
    home_garden_crops = [
        'TOMATOES', 'PEPPERS', 'LETTUCE', 'BEANS',
        'CUCUMBERS', 'SQUASH', 'STRAWBERRIES'
    ]
    if label_col:
        mask = df[label_col[0]].str.contains('|'.join(home_garden_crops), case=False, na=False)
        df_filtered = df[mask].copy()
        print(f'After home-garden filter: {df_filtered.shape}')
    else:
        df_filtered = df

    # Step 6: Engineer cost_per_yield if columns available
    if 'cost' in df_filtered.columns and 'yield' in df_filtered.columns:
        df_filtered['cost_per_yield'] = df_filtered['cost'] / (df_filtered['yield'] + 1e-9)

    print(f'Final shape: {df_filtered.shape}')
    return df_filtered

# Example:
# kaggle_clean = preprocess_kaggle_agri('data/raw/Crop_recommendation.csv')
# kaggle_clean.to_csv('data/processed/kaggle_agri_clean.csv', index=False)
print('Kaggle Agriculture pre-processing function ready ✓')

---
## 5. Indoor House Plants — Knowledge Base Pre-Processing

In [ ]:
# ── Indoor House Plants Dataset ──
# Download from: https://www.kaggle.com/datasets/andrewmvd/indoor-plant-disease
# Or: Kaggle indoor-house-plants-with-care-instructions

def preprocess_plants_knowledge(filepath):
    """Convert indoor plants dataset to a structured JSON knowledge base."""
    df = pd.read_csv(filepath)
    print(f'Raw shape: {df.shape}')
    print(df.columns.tolist())

    df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]

    # Map watering frequency text to numeric days
    watering_map = {
        'daily': 1,
        'every day': 1,
        'twice a week': 3,
        'every 3 days': 3,
        'weekly': 7,
        'once a week': 7,
        'every 2 weeks': 14,
        'biweekly': 14,
        'monthly': 30,
        'every month': 30,
    }

    def parse_watering(val):
        if pd.isna(val): return None
        val_lower = str(val).lower().strip()
        for key, days in watering_map.items():
            if key in val_lower:
                return days
        return None

    if 'watering' in df.columns:
        df['watering_days'] = df['watering'].apply(parse_watering)

    # Tag outdoor-applicable plants
    outdoor_friendly = [
        'tomato', 'pepper', 'basil', 'mint', 'rosemary', 'lavender',
        'thyme', 'sage', 'parsley', 'chives', 'lettuce', 'spinach'
    ]
    if 'common_name' in df.columns:
        df['applicable_to_outdoor'] = df['common_name'].str.lower().apply(
            lambda x: any(o in str(x) for o in outdoor_friendly))

    # Export as JSON knowledge base
    knowledge_base = df.to_dict(orient='records')
    output_path = PROCESSED_DIR / 'plant_knowledge_base.json'
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump(knowledge_base, f, indent=2, default=str)
    print(f'Knowledge base saved: {output_path} ({len(knowledge_base)} entries)')
    return df

# Example:
# plants_clean = preprocess_plants_knowledge('data/raw/indoor_plants.csv')
print('Indoor Plants knowledge base function ready ✓')

---
## 6. Data Quality Summary Report

In [ ]:
# ── Data Quality Report ──
# Run this after all pre-processing to get a summary for Prof. Thullen

def data_quality_report(df, dataset_name):
    """Print a data quality summary for any cleaned DataFrame."""
    print(f'\n{'='*50}')
    print(f'DATA QUALITY REPORT: {dataset_name}')
    print(f'{'='*50}')
    print(f'Shape:           {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'Missing values:  {df.isnull().sum().sum():,} total')
    print(f'Duplicate rows:  {df.duplicated().sum():,}')
    print(f'Memory usage:    {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
    print(f'\nColumn types:')
    print(df.dtypes.value_counts().to_string())
    print(f'\nMissing % per column (top 5):')
    missing = (df.isnull().mean() * 100).sort_values(ascending=False)
    print(missing[missing > 0].head(5).to_string())
    print()

# Example:
# data_quality_report(usda_clean, 'USDA NASS Quick Stats')
# data_quality_report(weather_clean, 'Open-Meteo Weather')
# data_quality_report(kaggle_clean, 'Kaggle Agriculture')
print('Data quality report function ready ✓')
print('\n✅ All pre-processing functions loaded. Set your data paths and API keys, then run each section.')